# How to use rugo with Parquet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/parquet.ipynb)

Read a Parquet file, look at its schema, read only the columns and rows you need, load it into pandas, and write Parquet back out - with rugo, which needs no PyArrow, pandas or NumPy of its own.

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo pandas

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.parquet"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.parquet", "space_missions.parquet")
print(f"space_missions.parquet: {os.path.getsize('space_missions.parquet'):,} bytes")

## Read a file

`read_parquet` is a context manager that yields **morsels** - batches of typed columns, one per row group.

In [ ]:
from rugo import parquet

with parquet.read_parquet("space_missions.parquet") as reader:
    for morsel in reader:
        print(morsel)

## Look at the schema without reading the data

`read_metadata` reads only the file footer: the row count and the column types.

In [ ]:
meta = parquet.read_metadata("space_missions.parquet")
print(meta.num_rows, "rows")
for column in meta.schema_columns:
    print(f"{column.name:16} {column.logical_type}")

## Read only some columns

Name the columns you want; the others are never decoded.

In [ ]:
with parquet.read_parquet("space_missions.parquet", columns=["Company", "Mission", "Price"]) as reader:
    for morsel in reader:
        print(morsel)

## Filter rows as you read

`predicates` is a list of `(column, operator, value)`. Row groups that can't match are skipped using the file's statistics, then the remaining rows are filtered exactly. Operators: `== != < <= > >= in "not in"`.

In [ ]:
with parquet.read_parquet(
    "space_missions.parquet",
    columns=["Company", "Mission", "Price"],
    predicates=[("Company", "==", "SpaceX"), ("Price", "<", 60.0)],
) as reader:
    for morsel in reader:
        print(morsel.num_rows, "rows")
        print(morsel)

## Get Python values

A morsel holds typed columns. `column(name).to_pylist()` turns one into a Python list.

In [ ]:
with parquet.read_parquet("space_missions.parquet", columns=["Mission"]) as reader:
    missions = [name for morsel in reader for name in morsel.column("Mission").to_pylist()]
print(missions[:5])

## Load a file into pandas

Rugo doesn't need pandas, but it's easy to hand data over: each column's `to_pylist()` gives the values, and a dict of lists is a DataFrame. This little helper does it for every morsel a reader yields.

In [ ]:
import pandas as pd


# Every morsel a rugo reader yields, as one pandas DataFrame.
def to_pandas(reader):
    frames = [
        pd.DataFrame({name.decode(): morsel.column(name).to_pylist() for name in morsel.column_names})
        for morsel in reader
    ]
    return pd.concat(frames, ignore_index=True)

In [ ]:
with parquet.read_parquet("space_missions.parquet", predicates=[("Company", "==", "SpaceX")]) as reader:
    df = to_pandas(reader)

df.head()

## Write a file

`write_parquet` turns a morsel into Parquet bytes (zstd-compressed by default).

In [ ]:
with parquet.read_parquet("space_missions.parquet", predicates=[("Mission_Status", "==", "Success")]) as reader:
    successes = next(iter(reader))

with open("successes.parquet", "wb") as f:
    f.write(parquet.write_parquet(successes))

print(parquet.read_metadata("successes.parquet").num_rows, "rows written")

## Write a big file a batch at a time

`open_parquet_writer` writes one row group per `write_row_group` call and passes the bytes to your sink as it goes, so memory stays flat however large the output.

In [ ]:
with open("copy.parquet", "wb") as f:
    with parquet.open_parquet_writer(f.write) as writer:
        with parquet.read_parquet("space_missions.parquet") as reader:
            for morsel in reader:
                writer.write_row_group(morsel)

print(parquet.read_metadata("copy.parquet").num_rows, "rows written")

## Convert to another format

Rugo's readers and writers share the morsel, so converting is a read and a write. Here Parquet becomes CSV.

In [ ]:
from rugo import csv

with parquet.read_parquet("space_missions.parquet") as reader:
    with open("space_missions.csv", "wb") as f:
        for i, morsel in enumerate(reader):
            f.write(csv.write_csv(morsel, header=(i == 0)))

print(open("space_missions.csv").readline())